In [1]:
#impordid
import pandas as pd
from pandas import *
import sqlite3
import numpy as np
import csv
from collections import Counter

# andmebaasi failinimi, kust andmeid loetake
filename = "C:\\Users\\kertu.saul\\OneDrive - Eesti Keele Instituut\\Dokumendid\\doktoritoo\\ressurssid\\rektsioonid\\katrin\\andmebaasifailid\\v33_koondkorpus_transaktsioonid.db"

# andmebaasiga ühenduse loomine
conn = sqlite3.connect(filename)
cursor = conn.cursor()

In [3]:
#tagastab andmebaasist lemma, sõnavormi ja näitefraasi iga kasutaja määratud sõna kohta, mis kasutaja määratud verbiga esineb 
def kohad_naidetega(verbid):
    query = (f"SELECT lemma, transaction_row.form, phrase FROM `transaction_row` JOIN `transaction_head` ON transaction_head.id = transaction_row.head_id "
             f"WHERE verb IN ({','.join('?' for _ in verbid)}) AND transaction_row.deprel = 'obl' "
             f"AND transaction_row.feats LIKE ?")
    cursor.execute(query, verbid + ('%adit%',))
    kohad_naidetega = list(cursor.fetchall())
    return kohad_naidetega

In [4]:
#määrame verbid, mille alluvate seast sõnu ja näiteid otsitakse
verbid = ('laduma',)
naited = kohad_naidetega(verbid)
#paneme tulemused dataframei
naite_df = pd.DataFrame(naited, columns =['lemma', 'form', 'naitelause'])
#võtame iga sõna jaoks ainult esimese näite
naited_unique = naite_df.drop_duplicates(subset='lemma', keep='first')
naited_unique

,lemma,form,naitelause
0,lett,letti,Tanel laob ehte eest kuulekalt letti dollarit
1,kael,kaela,kaela laob riigivõim karistused
2,pea,pähe,laob Dan soravalt pähe õpitult ette
3,kott,kotti,tädi laob kraami ukse taga kotti ringi
6,põhi,põhja,kevadel ladus ta hooajale põhja tingimustes
9,seljakott,seljakotti,jagu ladunud vara vabatahtlikult seljakotti
11,selg,selga,Tüdruk laob ülekuti selga riided
16,rida,ritta,poiss ladus vabatahtlikult jalatsid ritta
18,süli,sülle,laob sülle pakki
21,trepikoda,trepikotta,Pliidipuud laovad nad vanaemale trepikotta


In [21]:
#salvestame csv faili 
teekond = 'C:\\Users\\kertu.saul\\OneDrive - Eesti Keele Instituut\\Dokumendid\\doktoritoo\\fyysilised_kohad\\syntax_experiments_semantic_labelling\\physical_location_labelling\\results\\muutmisconx_naited.csv'
naited_unique.to_csv(teekond)